In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install koreanize_matplotlib

In [ ]:
import pandas as pd

df = pd.read_csv('train.csv')
df[['영업장명', '메뉴명']] = df['영업장명_메뉴명'].str.split('_', expand=True)

print(df[['영업장명', '메뉴명']].head())

In [ ]:
df['영업장명'] = df['영업장명'].astype('category')
df['메뉴명'] = df['메뉴명'].astype('category')
df.info()

In [ ]:
num_menus = df['메뉴명'].nunique()
num_stores = df['영업장명'].nunique()

print(f"총 메뉴 종류: {num_menus}개")
print(f"총 영업장 수: {num_stores}개")
print("\n" + "="*30 + "\n")

In [ ]:
sales_description = df['매출수량'].describe()

print("전체 판매량 통계:")
print(sales_description)
print("\n" + "="*30 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib

target_menu = '짜장밥'
menu_df = df[df['메뉴명'] == target_menu]

plt.figure(figsize=(12, 6))
plt.plot(menu_df['영업일자'], menu_df['매출수량'])
plt.title(f"'{target_menu}' 판매량 시계열 그래프")
plt.xlabel("영업일자")
plt.ylabel("매출수량")
plt.grid(True)
plt.show()

In [ ]:
df['영업일자'] = pd.to_datetime(df['영업일자'])
df_resample = df.set_index('영업일자')
monthly_sales = df_resample['매출수량'].resample('M').sum()

print("--- 월별 판매량 합계 (앞 5개) ---")
print(monthly_sales.head())

plt.figure(figsize=(12, 6))
monthly_sales.plot(title='월별 전체 메뉴 판매량 합계')
plt.xlabel('월')
plt.ylabel('총 판매량')
plt.grid(True)
plt.show()


weekly_sales = df_resample['매출수량'].resample('W').sum()

print("\n--- 주별 판매량 합계 (앞 5개) ---")
print(weekly_sales.head())

plt.figure(figsize=(12, 6))
weekly_sales.plot(title='주별 전체 메뉴 판매량 합계')
plt.xlabel('주')
plt.ylabel('총 판매량')
plt.grid(True)
plt.show()

In [ ]:
df['요일'] = df['영업일자'].dt.dayofweek
print("요일 컬럼 생성 완료.")

df['주말여부'] = df['요일'].isin([5, 6])
print("주말여부 컬럼 생성 완료.")

!pip install holidays -q

import holidays

start_year = df['영업일자'].dt.year.min()
end_year = df['영업일자'].dt.year.max()

kr_holidays = holidays.KR(years=range(start_year, end_year + 1))

df['공휴일여부'] = df['영업일자'].isin(kr_holidays)
print("공휴일여부 컬럼 생성 완료.")
df['월'] = df['영업일자'].dt.month
df['주차'] = df['영업일자'].dt.isocalendar().week

print("월, 주차 컬럼 생성 완료.")
print("\n--- 파생 변수 추가 후 데이터 (앞 5개) ---")
print(df.head())

In [ ]:
print("--- 컬럼별 결측치 개수 ---")
print(df.isnull().sum())
print("\n" + "="*40 + "\n")

negative_sales = df[df['매출수량'] < 0]
print(f"--- 판매량이 0보다 작은 데이터 ({len(negative_sales)}건) ---")

if not negative_sales.empty:
    print(negative_sales)
else:
    print("판매량이 0보다 작은 데이터는 없습니다.")

print("\n" + "="*40 + "\n")

q99 = df['매출수량'].quantile(0.99)
high_sales = df[df['매출수량'] > q99]
print(f"--- 판매량 상위 1% 이상 데이터 ({len(high_sales)}건) ---")
print(f"(매출수량 > {q99:.2f} 인 경우)")
print(high_sales.head())
print("\n" + "="*40 + "\n")


zero_sales_per_menu = df[df['매출수량'] == 0].groupby('메뉴명').size().sort_values(ascending=False)
print("--- 판매량이 0인 데이터가 많은 메뉴 (상위 10개) ---")
print(zero_sales_per_menu.head(10))
print("\n" + "="*40 + "\n")



num_negatives = len(df[df['매출수량'] < 0])
if num_negatives > 0:
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    print(f"--- 처리 완료: {num_negatives}건의 음수 판매량을 0으로 수정했습니다. ---")
else:
    print("--- 처리할 음수 판매량이 없습니다. ---")

In [ ]:
file_name = 'processed_train.csv'
df.to_csv(file_name, index=False)

print(f"--- 전처리 완료된 데이터가 '{file_name}'으로 저장되었습니다. ---")
print("Colab의 왼쪽 파일 탐색기에서 새로고침하면 파일을 확인할 수 있습니다.")

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("--- 이동 평균 특징 생성 중... ---")
df['7일_이동평균'] = df.groupby(['영업장명', '메뉴명'])['매출수량'].transform(lambda x: x.rolling(window=7,min_periods=1).mean())
df['28일_이동평균'] = df.groupby(['영업장명', '메뉴명'])['매출수량'].transform(lambda x: x.rolling(window=28,min_periods=1).mean())

print("이동 평균 특징 생성 완료.")

cols_to_fill = ['7일_이동평균', '28일_이동평균']
df[cols_to_fill] = df[cols_to_fill].fillna(0)
print("NaN 값 처리 완료.")
print("\n" + "="*40 + "\n")

print("--- 라벨 인코딩 진행 중... ---")
df['영업장명_인코딩'] = df['영업장명'].cat.codes
df['메뉴명_인코딩'] = df['메뉴명'].cat.codes
print("라벨 인코딩 완료.")
print("\n" + "="*40 + "\n")


print("--- Feature Engineering 추가 후 데이터 정보 ---")
print(df.info())
print("\n--- 데이터 샘플 (앞 5개) ---")
print(df.head())

In [ ]:
import numpy as np

df['매출수량_로그'] = np.log1p(df['매출수량'])
print("--- '매출수량_로그' 컬럼 추가 완료 ---")
print(df[['매출수량', '매출수량_로그']].head())

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

features = [
    '매출수량_로그',
    '요일',
    '주말여부',
    '공휴일여부',
    '월',
    '주차',
    '7일_이동평균',
    '28일_이동평균',
    '영업장명_인코딩',
    '메뉴명_인코딩'
    ]

X_data = []
y_data = []

INPUT_DAYS = 28
OUTPUT_DAYS = 7

grouped = df.groupby(['영업장명_인코딩', '메뉴명_인코딩'])

for _, group in tqdm(grouped, desc="학습 데이터 생성 중"):

    if len(group) < INPUT_DAYS + OUTPUT_DAYS:
        continue

    for i in range(len(group) - INPUT_DAYS - OUTPUT_DAYS + 1):
        X_sample = group[features].iloc[i : i + INPUT_DAYS].values

        y_sample = group['매출수량_로그'].iloc[i + INPUT_DAYS : i + INPUT_DAYS + OUTPUT_DAYS].values

        X_data.append(X_sample.flatten())
        y_data.append(y_sample)

X_train = np.array(X_data)
y_train = np.array(y_data)


print("\n--- 생성된 학습 데이터의 형태(shape) ---")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

In [ ]:
!pip install optuna -q

In [ ]:
#6
import optuna
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)
#X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.1, shuffle=False, random_state=42)


def objective(trial):
    params = {
        'objective': 'regression_l1', # MAE를 손실 함수로 사용
        'metric': 'rmse',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        }

    lgbm = lgb.LGBMRegressor(**params)
    model = MultiOutputRegressor(lgbm)

    rmse_scores = []

    for train_index, val_index in tscv.split(X_train) :
        X_train_split, X_val = X_train[train_index], X_train[val_index]
        y_train_split, y_val = y_train[train_index], y_train[val_index]

        model.fit(X_train_split, y_train_split)
        y_pred = model.predict(X_val)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_scores)
    #model.fit(X_train_split, y_train_split)
    #y_pred = model.predict(X_val)
    #rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    #return rmse

print("--- 하이퍼파라미터 튜닝 시작... (n_trials 횟수에 따라 시간이 많이 소요될 수 있습니다) ---")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20) # 50번의 다른 조합을 테스트합니다.


print("\n--- 튜닝 완료! ---")
print(f"최적의 RMSE 점수: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)
print("\n" + "="*40 + "\n")

best_params = study.best_params
final_lgbm = lgb.LGBMRegressor(objective='regression_l1', metric='rmse', random_state=42, **best_params)
multi_output_model = MultiOutputRegressor(final_lgbm) # multi_output_model 변수를 최종 모델로 업데이트

print("--- 최적의 파라미터로 최종 모델 학습 중... ---")
multi_output_model.fit(X_train, y_train)
print("--- 최종 모델 학습 완료! ---")

"""import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import numpy as np

X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.1, shuffle=False, random_state=42)

print("--- 데이터 분할 완료 ---")
print(f"학습용 데이터: X={X_train_split.shape}, y={y_train_split.shape}")
print(f"검증용 데이터: X={X_val.shape}, y={y_val.shape}")
print("\n" + "="*40 + "\n")

print("--- 모델 학습 시작... (시간이 조금 걸릴 수 있습니다)---")
lgbm = lgb.LGBMRegressor(random_state=42)
multi_output_model = MultiOutputRegressor(lgbm)
multi_output_model.fit(X_train_split, y_train_split)
print("--- 모델 학습 완료 ---")
print("\n" + "="*40 + "\n")

y_pred = multi_output_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"--- 모델 성능 평가 (Validation RMSE) ---")
print(f"RMSE: {rmse:.4f}")
print("\n" + "="*40 + "\n")

sample_idx = 0
sample_actual = y_val[sample_idx]
sample_pred = y_pred[sample_idx]
sample_actual_orig = np.expm1(sample_actual)
sample_pred_orig = np.expm1(sample_pred)

plt.figure(figsize=(12, 6))
plt.plot(range(1, 8), sample_actual_orig, 'o-', label='실제값 (Actual)')
plt.plot(range(1, 8), sample_pred_orig, 'o--', label='예측값 (Predicted)')
plt.title(f'검증 샘플 {sample_idx}번: 7일간 판매량 예측 결과 비교')
plt.xlabel('일 (Day)')
plt.ylabel('판매량')
plt.legend()
plt.grid(True)
plt.show()"""

In [ ]:
#7
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
print("\n" + "="*40 + "\n")

    # 실제 파일 이름에 맞춰 _ 또는 띄어쓰기를 사용해주세요
test_files = glob.glob('/content/drive/MyDrive/LG_COLAB_DATA/TEST_*.csv')
test_files.sort()

print(f"--- 총 {len(test_files)}개의 테스트 파일 감지 ---")
print(test_files)
print("\n" + "="*40 + "\n")

submission_df = pd.DataFrame()
store_map = dict(zip(df['영업장명'], df['영업장명_인코딩']))
menu_map = dict(zip(df['메뉴명'], df['메뉴명_인코딩']))

for test_file in tqdm(test_files, desc="테스트 파일 예측 중"):
    test_df = pd.read_csv(test_file)

    test_df[['영업장명', '메뉴명']] = test_df['영업장명_메뉴명'].str.split('_', expand=True)
    test_df['영업일자'] = pd.to_datetime(test_df['영업일자'])
    test_df['요일'] = test_df['영업일자'].dt.dayofweek
    test_df['주말여부'] = test_df['요일'].isin([5, 6])
    start_year = test_df['영업일자'].dt.year.min()
    end_year = test_df['영업일자'].dt.year.max()
    kr_holidays = holidays.KR(years=range(start_year, end_year + 1))
    test_df['공휴일여부'] = test_df['영업일자'].isin(kr_holidays)
    test_df['월'] = test_df['영업일자'].dt.month
    test_df['주차'] = test_df['영업일자'].dt.isocalendar().week
    test_df['매출수량_로그'] = np.log1p(test_df['매출수량'])
    test_df['7일_이동평균'] = test_df.groupby(['영업장명', '메뉴명'])['매출수량_로그'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
    test_df['28일_이동평균'] = test_df.groupby(['영업장명', '메뉴명'])['매출수량_로그'].transform(lambda x: x.rolling(window=28, min_periods=1).mean())
    cols_to_fill = ['7일_이동평균', '28일_이동평균']
    test_df[cols_to_fill] = test_df[cols_to_fill].fillna(0)
    test_df['영업장명_인코딩'] = test_df['영업장명'].map(store_map)
    test_df['메뉴명_인코딩'] = test_df['메뉴명'].map(menu_map)
    test_df.fillna(-1, inplace=True)

    test_grouped = test_df.groupby(['영업장명_인코딩', '메뉴명_인코딩'])

    for _, group in test_grouped:
        X_test = group[features].values.flatten().reshape(1, -1)

        if X_test.shape[1] != 280:
            continue

        y_pred_log = multi_output_model.predict(X_test)
        y_pred_orig = np.expm1(y_pred_log)

        test_file_id = test_file.split('/')[-1].split('.')[0]

        temp_submission = pd.DataFrame({
                'test_id': test_file_id,
                'day_offset': range(1, 8),
                '영업장명_메뉴명': group['영업장명_메뉴명'].iloc[0],
                '예측매출수량': y_pred_orig.flatten()
                })

        submission_df = pd.concat([submission_df, temp_submission])

print("\n--- 모든 테스트 파일 예측 완료 (ID 정보 포함) ---")
print(submission_df.head(10))


In [ ]:
print("--- 7단계 실행 후 submission_df가 비어있는지 확인 ---")
print(submission_df.empty)
print("\n--- submission_df 내용 확인 (상위 5줄) ---")
print(submission_df.head())

In [ ]:
   print(submission_df.head())

In [ ]:
#8
import pandas as pd

if submission_df.empty or 'test_id' not in submission_df.columns:
    print("!!! 오류: 8단계를 실행하기 전에, ID 정보가 포함된 최신 7단계 코드를 먼저 실행해야 합니다. !!!")

else:
    submission_df['ID'] = submission_df['test_id'] + '+' + submission_df['day_offset'].astype(str) + '일'
    pivoted_df = submission_df.pivot_table(index='ID', columns='영업장명_메뉴명', values='예측매출수량')

    sample_path ='/content/drive/MyDrive/LG_COLAB_DATA/sample_submission.csv'
    sample_submission = pd.read_csv(sample_path)

    final_submission = pivoted_df.reindex(columns=sample_submission.columns[1:])

    final_submission.fillna(0, inplace=True)

    for col in final_submission.columns:
        final_submission[col] = final_submission[col].astype(int)

    final_submission.reset_index(inplace=True)
    final_submission.rename(columns={'ID': '영업일자'}, inplace=True)

    final_submission.to_csv('submission.csv', index=False)

    print("--- 최종 제출 파일 'submission.csv' 생성 완료! ---")
    print("--- 생성된 파일 샘플 (앞 5개) ---")
    print(final_submission.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/LG_COLAB_DATA/